# SARIMAX estadual — tipologias estimáveis

Esta análise foi executada no script `src/sarimax_revisao_final.py`. O notebook examina e interpreta as saídas salvas. Para reestimar os modelos, execute o script conforme a seção de reprodução. As seleções não utilizam a amostra de teste.

## 1. Protocolo de estimação

Resposta: `100 Δlog(I)`. Exógenas: `Δlog(1+D)`. SARIMA e SARIMAX usam a mesma ordem e sazonalidade anual `s=12` no par; a escolha é feita no treino. Avaliação condicional de um passo com `x_t` conhecida.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json
root=Path.cwd()
if not (root/'outputs'/'tables').exists(): root=root.parent
tables=root/'outputs'/'tables'/'sarimax_revisao_20261019'


In [2]:
df=pd.read_csv(tables/'estadual_comparacao_modelos.csv')
audit=pd.read_csv(tables/'estadual_auditoria_posto.csv')
coef=pd.read_csv(tables/'estadual_coeficientes.csv')
summary=json.loads((tables/'estadual_resumo.json').read_text(encoding='utf-8'))
print('Resumo:', summary)
print('Postos originais:', audit.groupby('periodo')[['posto_treino','posto_completo']].agg(['min','max']).to_string())

Resumo: {'amostra': '2013-2019/2020 e 2013-2024', 'modo': 'estadual', 'n_modelos': 54, 'n_comparaveis': 53, 'n_melhor_rmse': 14, 'n_melhor_bic': 0, 'n_LB_sarimax_nao_rejeita': 41, 'n_LB_sarima_nao_rejeita': 44, 'n_rejeicoes_conjunto_BH': 0, 'n_rejeicoes_coef_global_BH': 2, 'mediana_delta_RMSE_pct': 7.194923074761506, 'criterio_selecao': 'apenas treino: suporte>=3 meses, posto completo, cond<=1000 e cap 9 ou 12 (estaduais)', 'exog_t': 'Δlog(1+D), contemporâneo, conhecido no teste', 'y': '100 Δlog(CI/CA), Brasil calculado por soma das carteiras'}
Postos originais:         posto_treino     posto_completo    
                 min max            min max
periodo                                    
Pré                2  15              3  15
Total              5  16              7  16


## 2. Resultados e comparação

Δ RMSE negativo indica melhora do SARIMAX. BIC comparado apenas para modelos ajustados no mesmo treino.

In [3]:
cols=['periodo','unidade','n_exog_incluidas','sazonal','rmse_sarima','rmse_sarimax','variacao_rmse_pct','delta_bic_treino','p_conjunto_treino','q_conjunto_bh','lb_sarimax_treino']
print(df[cols].to_string(index=False))
print('Modelos comparáveis:',df.rmse_sarimax.notna().sum(),'Melhora RMSE:',(df.variacao_rmse_pct<0).sum(),'Melhora BIC:',(df.delta_bic_treino<0).sum())

periodo unidade  n_exog_incluidas       sazonal  rmse_sarima  rmse_sarimax  variacao_rmse_pct  delta_bic_treino  p_conjunto_treino  q_conjunto_bh  lb_sarimax_treino
    Pré      AC                 3 (1, 0, 0, 12)     2.729315      2.685643          -1.600125         11.967085           0.834367       0.964119           0.256519
    Pré      AL                 2 (1, 0, 0, 12)     2.303732      2.500009           8.519950          5.277158           0.194351       0.875374           0.180342
    Pré      AM                 8 (1, 0, 0, 12)     2.002416      2.314547          15.587692         23.965908           0.248088       0.875374           0.009866
    Pré      AP                 3 (1, 0, 0, 12)     1.571773      1.700196           8.170589          8.510548           0.228971       0.875374           0.548271
    Pré      BA                 8 (1, 0, 0, 12)     2.328861      2.754358          18.270624         15.991130           0.019621       0.582991           0.200203
    Pré   

## 3. Evidências e limites

Os p-valores são condicionais à seleção do modelo e não representam causalidade. Os regressors `x_t` contemporâneos são conhecidos no teste; a previsão é condicional. Verificar o Ljung–Box e a família de multiplicidade ao interpretar efeitos individuais.

In [4]:
if 'q_bh_global' in coef.columns:
    print(coef.loc[coef.q_bh_global<.05,['periodo','unidade','tipologia','coef_original','p','q_bh_global']].to_string(index=False))
else: print('Sem coeficientes elegíveis ao BH global')

periodo unidade       tipologia  coef_original        p  q_bh_global
    Pré      PE tipo_enxurradas       1.159152 0.000116     0.022858
  Total      PE tipo_enxurradas       0.829787 0.000064     0.022858


## 4. Reprodução integral

No terminal do projeto: `python src/sarimax_revisao_final.py estadual`. Consulte `docs/sarimax_revisao_estadual_nacional_20261019.md` para definições, seleção de suporte, limitações e conclusões.